In [1]:
import pandas as pd;
import numpy as np;
import matplotlib.pyplot as plt;
import seaborn as sns;

# Test Case 11

In [17]:
quality = pd.read_csv("quality_inspection_cleaned_T18.csv")

In [18]:
severity_map = {
    "Scratch": 3,
    "Dent": 5,
    "Paint Defect": 6,
    "Weld Failure": 9,
    "Engine Failure": 10,
    "Brake Failure": 10,
    "Electrical Fault": 8,
    "Misalignment": 7
}

In [19]:
quality["Severity"] = (
    quality["Defect"]
    .map(severity_map)
    .fillna(5)
)

In [20]:
occurrence = (
    quality.groupby("Defect_Code")
    .size()
    .reset_index(name="Count")
)

In [21]:
max_count = occurrence["Count"].max()

occurrence["Occurrence"] = (
    occurrence["Count"] / max_count * 10
).round().clip(1,10)

In [22]:
quality = quality.merge(
    occurrence[["Defect_Code","Occurrence"]],
    on="Defect_Code",
    how="left"
)

In [23]:
quality["Detection"] = np.where(
    quality["Inspection_Source"].str.upper()=="AUTOMATED",
    2,
    6
)

In [24]:
quality["RPN"] = (
    quality["Severity"] *
    quality["Occurrence"] *
    quality["Detection"]
)

In [25]:
quality["Risk_Level"] = np.where(
    quality["RPN"]>=300,
    "High",
    np.where(
        quality["RPN"]>=100,
        "Medium",
        "Low"
    )
)

In [26]:
pfmea = quality[[
    "Defect_Code",
    "Defect",
    "Severity",
    "Occurrence",
    "Detection",
    "RPN",
    "Risk_Level"
]]

In [27]:
pfmea = pfmea.sort_values(
    by="RPN",
    ascending=False
)

In [28]:
print(pfmea.head(20))

     Defect_Code  Defect  Severity  Occurrence  Detection    RPN Risk_Level
3985        NONE  REJECT       5.0        10.0          6  300.0       High
3990        NONE    NONE       5.0        10.0          6  300.0       High
3999        NONE      OK       5.0        10.0          6  300.0       High
3959        NONE  REJECT       5.0        10.0          6  300.0       High
3962        NONE      OK       5.0        10.0          6  300.0       High
3936        NONE      OK       5.0        10.0          6  300.0       High
3944        NONE  REJECT       5.0        10.0          6  300.0       High
3945        NONE  REJECT       5.0        10.0          6  300.0       High
3948        NONE      OK       5.0        10.0          6  300.0       High
3891        NONE  REPAIR       5.0        10.0          6  300.0       High
3896        NONE    NONE       5.0        10.0          6  300.0       High
3898        NONE    NONE       5.0        10.0          6  300.0       High
3903        

In [29]:
pfmea.to_csv("PFMEA_Risk_Analysis.csv", index=False)

print("\nPFMEA Risk Analysis Completed Successfully!")


PFMEA Risk Analysis Completed Successfully!


#test case = 12
